# 11｜Final Findings & Project Summary

這一本不再建立新的模型或 threshold。

它只做三件事：

1. 讀回 02～10 已經輸出的正式 artifacts。
2. 檢查這些 artifacts 的 lineage 有沒有對上。
3. 把最新結果整理成 Final Findings。

這一版另外正式接入：

- 04 的 **Lot-level Bootstrap Uncertainty**
- 08 的 **Process + Sequence Nested Ablation**

09 / 10 仍然沿用 08 的 **Primary Process-only Nested OOF**，
因為 Explainability、Model Error 與 Anomaly lineage 原本就是建立在這個 primary model 上。


## 1. 讀入 02～10 的正式 artifacts

先把 Final Report 會用到的資料一次讀進來。

這裡不做 fallback，也不會在缺檔時偷偷重算。如果少任何一個必要 artifact，我寧願直接停掉，因為 Final Report 最怕的就是把不同版本的結果混在一起。

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import r2_score

pd.set_option("display.max_columns", 240)
pd.set_option("display.max_rows", 220)


def find_project_root(start=None):
    """Find the project root by walking upward from the current directory."""
    start = Path.cwd().resolve() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "notebooks").exists()
            and (candidate / "data").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Project root not found. Expected a parent directory "
        "containing both notebooks/ and data/."
    )


CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = find_project_root(CURRENT_DIR)


PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
).resolve()

paths = {
    # 02 / 03
    "quality":
        PROCESSED_DATA_DIR
        / "wafer_quality_summary.csv",

    "lot_quality":
        PROCESSED_DATA_DIR
        / "lot_quality_summary.csv",

    "radial":
        PROCESSED_DATA_DIR
        / "radial_summary.csv",

    "lot_sequence":
        PROCESSED_DATA_DIR
        / "lot_sequence_summary.csv",

    "process_summary":
        PROCESSED_DATA_DIR
        / "process_wafer_summary.csv",

    "process_sequence_summary":
        PROCESSED_DATA_DIR
        / "process_sequence_summary.csv",

    # 04
    "mean_candidates":
        PROCESSED_DATA_DIR
        / "mean_etch_exploratory_candidates.csv",

    "cv_candidates":
        PROCESSED_DATA_DIR
        / "cv_exploratory_candidates.csv",

    "mean_bootstrap":
        PROCESSED_DATA_DIR
        / "04_mean_etch_relationship_bootstrap.csv",

    "cv_bootstrap":
        PROCESSED_DATA_DIR
        / "04_cv_relationship_bootstrap.csv",

    # 05 / 06
    "spc_feature":
        PROCESSED_DATA_DIR
        / "spc_feature_summary.csv",

    "spc_wafer":
        PROCESSED_DATA_DIR
        / "spc_wafer_flag_summary.csv",

    "feature_sequence_comparison":
        PROCESSED_DATA_DIR
        / "06_feature_sequence_comparison.csv",

    "raw_pca_var":
        PROCESSED_DATA_DIR
        / "06_raw_pca_variance.csv",

    "residual_pca_var":
        PROCESSED_DATA_DIR
        / "06_residual_pca_variance.csv",

    # 08 primary model
    "baseline":
        PROCESSED_DATA_DIR
        / "08_baseline_metrics.csv",

    "outer":
        PROCESSED_DATA_DIR
        / "08_nested_outer_fold_results.csv",

    "oof":
        PROCESSED_DATA_DIR
        / "08_nested_oof_predictions.csv",

    "config":
        PROCESSED_DATA_DIR
        / "08_final_model_config.json",

    # 08 ablation
    "ablation_metrics":
        PROCESSED_DATA_DIR
        / "08_ablation_metrics.csv",

    "ablation_outer":
        PROCESSED_DATA_DIR
        / "08_process_plus_sequence_outer_fold_results.csv",

    "ablation_predictions":
        PROCESSED_DATA_DIR
        / "08_ablation_oof_predictions.csv",

    # 09
    "coef_stability":
        PROCESSED_DATA_DIR
        / "09_coefficient_stability.csv",

    "sensor_coef":
        PROCESSED_DATA_DIR
        / "09_sensor_coefficient_summary.csv",

    "sensor_perm":
        PROCESSED_DATA_DIR
        / "09_sensor_grouped_permutation_importance.csv",

    "error_by_lot":
        PROCESSED_DATA_DIR
        / "09_nested_error_by_lot.csv",

    # 10
    "residual_anomaly":
        PROCESSED_DATA_DIR
        / "10_residual_anomaly_scores.csv",

    "ood":
        PROCESSED_DATA_DIR
        / "10_cross_lot_ood_scores.csv",

    "ood_thresholds":
        PROCESSED_DATA_DIR
        / "10_ood_fold_thresholds.csv",

    "tri":
        PROCESSED_DATA_DIR
        / "10_anomaly_triangulation.csv",

    "lot":
        PROCESSED_DATA_DIR
        / "10_anomaly_by_lot.csv",

    "association":
        PROCESSED_DATA_DIR
        / "10_process_vs_model_error_association.csv",

    "priority":
        PROCESSED_DATA_DIR
        / "10_priority_cases.csv",

    "methodology":
        PROCESSED_DATA_DIR
        / "10_anomaly_methodology.json",
}

missing = [
    path
    for path in paths.values()
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "11 缺少必要 upstream artifacts：\n"
        + "\n".join(
            str(path)
            for path in missing
        )
    )


quality = pd.read_csv(
    paths["quality"]
)

lot_quality = pd.read_csv(
    paths["lot_quality"]
)

radial = pd.read_csv(
    paths["radial"]
)

lot_sequence = pd.read_csv(
    paths["lot_sequence"]
)

process_summary = pd.read_csv(
    paths["process_summary"]
)

process_sequence_summary = pd.read_csv(
    paths["process_sequence_summary"]
)

mean_candidates = pd.read_csv(
    paths["mean_candidates"]
)

cv_candidates = pd.read_csv(
    paths["cv_candidates"]
)

mean_bootstrap = pd.read_csv(
    paths["mean_bootstrap"]
)

cv_bootstrap = pd.read_csv(
    paths["cv_bootstrap"]
)

spc_feature = pd.read_csv(
    paths["spc_feature"]
)

spc_wafer = pd.read_csv(
    paths["spc_wafer"]
)

feature_sequence_comparison = pd.read_csv(
    paths["feature_sequence_comparison"]
)

raw_pca_var = pd.read_csv(
    paths["raw_pca_var"]
)

residual_pca_var = pd.read_csv(
    paths["residual_pca_var"]
)

baseline = pd.read_csv(
    paths["baseline"]
)

outer = pd.read_csv(
    paths["outer"]
)

nested_oof = pd.read_csv(
    paths["oof"]
)

ablation_metrics = pd.read_csv(
    paths["ablation_metrics"]
)

ablation_outer = pd.read_csv(
    paths["ablation_outer"]
)

ablation_predictions = pd.read_csv(
    paths["ablation_predictions"]
)

coef_stability = pd.read_csv(
    paths["coef_stability"]
)

sensor_coef = pd.read_csv(
    paths["sensor_coef"]
)

sensor_perm = pd.read_csv(
    paths["sensor_perm"]
)

error_by_lot = pd.read_csv(
    paths["error_by_lot"]
)

residual_anomaly = pd.read_csv(
    paths["residual_anomaly"]
)

cross_lot_ood = pd.read_csv(
    paths["ood"]
)

ood_fold_thresholds = pd.read_csv(
    paths["ood_thresholds"]
)

tri = pd.read_csv(
    paths["tri"]
)

lot_summary = pd.read_csv(
    paths["lot"]
)

association = pd.read_csv(
    paths["association"]
)

priority_cases = pd.read_csv(
    paths["priority"]
)

with open(
    paths["config"],
    "r",
    encoding="utf-8",
) as file:
    final_config = json.load(
        file
    )

with open(
    paths["methodology"],
    "r",
    encoding="utf-8",
) as file:
    anomaly_methodology = json.load(
        file
    )

print(
    "Final Report artifacts 載入完成"
)

print(
    "Process wafers：",
    len(tri),
)

print(
    "Quality-labeled / OOF wafers：",
    len(nested_oof),
)

print(
    "Lots：",
    tri["lot_number"].nunique(),
)

print(
    "04 Bootstrap tables：",
    len(mean_bootstrap),
    "/",
    len(cv_bootstrap),
)

print(
    "08 Ablation models：",
    len(ablation_metrics),
)


Final Report artifacts 載入完成
Process wafers： 96
Quality-labeled / OOF wafers： 88
Lots： 10
04 Bootstrap tables： 27 / 27
08 Ablation models： 4


### 這個輸出代表什麼？

這格確認 11 需要的新版 artifacts 都有成功讀進來。

實際讀到：

- Process wafers：**96**
- Quality-labeled / OOF wafers：**88**
- Lots：**10**
- 04 Bootstrap tables：**27 / 27**
- 08 Ablation models：**4**

所以 11 現在接到的是完整的 96 片 Process population，以及其中 88 片有 Quality label 和 OOF prediction 的資料。

另外，04 的 Mean / CV Bootstrap table 都各有 **27 個 features**，08 的 Ablation table 也有完整的 **4 種模型比較**。


## 2. Cross-Artifact Validation：先確認 02～10 真的是同一條資料流程

Final Report 最重要的不是表很多，而是前後不能互相矛盾。

所以我先驗幾條 lineage：

- Quality wafers 和 08 Nested OOF 必須是同一批 labeled wafers。
- 08 的 `nested_error` 必須真的等於 prediction − actual。
- 09 per-Lot error 必須能從 08 OOF 重新算回來。
- 10 的 Lot-level VM RMSE 必須和 08 OOF 一致。
- 06 residual PCA 和 10 residual anomaly 必須使用同一個 residual universe。
- 10 Pure Cross-Lot OOD 必須確認：
  - 只使用 07 process engineered features。
  - 不使用 Quality target。
  - 不使用 08 error。
  - 不使用 09 importance。
  - Inner calibration 和 final outer detector 使用同一套 **outer-training-only fixed feature universe**。

In [2]:
def assert_unique_key(
    dataframe,
    name,
):
    if (
        "experiment_key"
        not in dataframe.columns
    ):
        raise ValueError(
            f"{name} 缺少 experiment_key"
        )

    if dataframe[
        "experiment_key"
    ].duplicated().any():
        raise ValueError(
            f"{name} experiment_key 有重複"
        )


for name, dataframe in {
    "quality": quality,
    "nested_oof": nested_oof,
    "ablation_predictions":
        ablation_predictions,
    "residual_anomaly":
        residual_anomaly,
    "cross_lot_ood":
        cross_lot_ood,
    "triangulation":
        tri,
}.items():
    assert_unique_key(
        dataframe,
        name,
    )


# 02 -> 08 wafer identity
if (
    set(
        quality["experiment_key"]
    )
    != set(
        nested_oof["experiment_key"]
    )
):
    raise ValueError(
        "Quality / Nested OOF wafer identity 不一致"
    )


if (
    set(
        nested_oof["experiment_key"]
    )
    != set(
        ablation_predictions[
            "experiment_key"
        ]
    )
):
    raise ValueError(
        "Primary OOF / Ablation wafer identity 不一致"
    )


if not set(
    quality["experiment_key"]
).issubset(
    set(
        tri["experiment_key"]
    )
):
    raise ValueError(
        "Labeled wafers 不是 Process population 的子集"
    )


if (
    set(
        residual_anomaly[
            "experiment_key"
        ]
    )
    != set(
        tri["experiment_key"]
    )
):
    raise ValueError(
        "Residual anomaly population 與 triangulation 不一致"
    )


if (
    set(
        cross_lot_ood[
            "experiment_key"
        ]
    )
    != set(
        tri["experiment_key"]
    )
):
    raise ValueError(
        "Cross-Lot OOD population 與 triangulation 不一致"
    )


# 04 candidate -> bootstrap lineage
for candidate_df, bootstrap_df, name in [
    (
        mean_candidates,
        mean_bootstrap,
        "Mean Si Etch",
    ),
    (
        cv_candidates,
        cv_bootstrap,
        "CV Si Etch",
    ),
]:
    if not set(
        candidate_df["feature"]
    ).issubset(
        set(
            bootstrap_df["feature"]
        )
    ):
        raise ValueError(
            f"04 {name} candidates 不在 bootstrap table"
        )


# 08 primary prediction error formula
calc_error = (
    nested_oof[
        "nested_process_pred"
    ].to_numpy(float)
    - nested_oof[
        "actual_mean_si_etch"
    ].to_numpy(float)
)

if not np.allclose(
    calc_error,
    nested_oof[
        "nested_error"
    ].to_numpy(float),
    atol=1e-12,
    rtol=0,
):
    raise ValueError(
        "08 nested_error formula 不一致"
    )

if not np.allclose(
    np.abs(calc_error),
    nested_oof[
        "abs_nested_error"
    ].to_numpy(float),
    atol=1e-12,
    rtol=0,
):
    raise ValueError(
        "08 abs_nested_error formula 不一致"
    )


# 08 primary -> ablation lineage
primary_for_ablation = (
    nested_oof[
        [
            "experiment_key",
            "actual_mean_si_etch",
            "nested_process_pred",
        ]
    ]
    .merge(
        ablation_predictions[
            [
                "experiment_key",
                "actual_mean_si_etch",
                "process_only_pred",
                "process_plus_sequence_pred",
            ]
        ],
        on="experiment_key",
        how="inner",
        suffixes=(
            "_primary",
            "_ablation",
        ),
        validate="one_to_one",
    )
)

if not np.allclose(
    primary_for_ablation[
        "actual_mean_si_etch_primary"
    ].to_numpy(float),
    primary_for_ablation[
        "actual_mean_si_etch_ablation"
    ].to_numpy(float),
    atol=1e-12,
    rtol=0,
):
    raise ValueError(
        "08 Primary / Ablation target 不一致"
    )

if not np.allclose(
    primary_for_ablation[
        "nested_process_pred"
    ].to_numpy(float),
    primary_for_ablation[
        "process_only_pred"
    ].to_numpy(float),
    atol=1e-12,
    rtol=0,
):
    raise ValueError(
        "08 Primary Process-only prediction / Ablation process_only_pred 不一致"
    )

if not np.isfinite(
    primary_for_ablation[
        "process_plus_sequence_pred"
    ].to_numpy(float)
).all():
    raise ValueError(
        "08 Process + Sequence prediction 有 NaN / Inf"
    )


# Rebuild per-Lot primary VM error from 08 OOF
recalc_lot = (
    nested_oof.groupby(
        "lot_number",
        as_index=False,
    )
    .agg(
        n_wafers=(
            "experiment_key",
            "size",
        ),
        rmse=(
            "nested_error",
            lambda values:
                float(
                    np.sqrt(
                        np.mean(
                            np.asarray(
                                values,
                                float,
                            )
                            ** 2
                        )
                    )
                ),
        ),
        mae=(
            "abs_nested_error",
            "mean",
        ),
        max_abs_error=(
            "abs_nested_error",
            "max",
        ),
    )
    .sort_values(
        "lot_number"
    )
    .reset_index(
        drop=True
    )
)


# 08 -> 09 lineage
err_check = (
    error_by_lot
    .sort_values(
        "lot_number"
    )
    .reset_index(
        drop=True
    )
)

for column in [
    "n_wafers",
    "rmse",
    "mae",
    "max_abs_error",
]:
    if not np.allclose(
        recalc_lot[
            column
        ].to_numpy(float),
        err_check[
            column
        ].to_numpy(float),
        atol=1e-12,
        rtol=0,
    ):
        raise ValueError(
            "08 → 09 Lot error lineage 不一致："
            f"{column}"
        )


# 08 -> 10 lineage
lot_vm = (
    lot_summary[
        [
            "lot_number",
            "vm_rmse",
            "vm_max_abs_error",
        ]
    ]
    .sort_values(
        "lot_number"
    )
    .reset_index(
        drop=True
    )
)

if not np.allclose(
    lot_vm[
        "vm_rmse"
    ].to_numpy(float),
    recalc_lot[
        "rmse"
    ].to_numpy(float),
    atol=1e-12,
    rtol=0,
):
    raise ValueError(
        "08 → 10 Lot VM RMSE lineage 不一致"
    )

if not np.allclose(
    lot_vm[
        "vm_max_abs_error"
    ].to_numpy(float),
    recalc_lot[
        "max_abs_error"
    ].to_numpy(float),
    atol=1e-12,
    rtol=0,
):
    raise ValueError(
        "08 → 10 Lot VM max error lineage 不一致"
    )


# 06 -> 10 residual lineage
n_resid_06 = len(
    residual_pca_var
)

n_resid_10 = int(
    anomaly_methodology[
        "residual_feature_count"
    ]
)

if n_resid_06 != n_resid_10:
    raise ValueError(
        "06 / 10 residual feature count 不一致："
        f"{n_resid_06} vs {n_resid_10}"
    )


pca90_06 = int(
    residual_pca_var.loc[
        residual_pca_var[
            "cumulative_variance_ratio"
        ]
        >= 0.90,
        "component",
    ].iloc[0]
)

pca90_10 = int(
    anomaly_methodology[
        "residual_pca_components_90pct"
    ]
)

if pca90_06 != pca90_10:
    raise ValueError(
        "06 / 10 residual PCA 90% components 不一致："
        f"{pca90_06} vs {pca90_10}"
    )


# 10 Pure OOD source separation
if (
    anomaly_methodology.get(
        "pure_cross_lot_ood_uses_quality_target"
    )
    is not False
):
    raise ValueError(
        "Pure OOD 不應使用 Quality target"
    )

if (
    anomaly_methodology.get(
        "pure_cross_lot_ood_uses_notebook08_error"
    )
    is not False
):
    raise ValueError(
        "Pure OOD 不應使用 08 prediction error"
    )

if (
    anomaly_methodology.get(
        "pure_cross_lot_ood_uses_notebook09_importance"
    )
    is not False
):
    raise ValueError(
        "Pure OOD 不應使用 09 importance"
    )

if (
    anomaly_methodology.get(
        "pure_cross_lot_ood_fixed_outer_feature_universe"
    )
    is not True
):
    raise ValueError(
        "Pure OOD 尚未使用 fixed outer feature universe"
    )

if not ood_fold_thresholds[
    "calibration_uses_fixed_outer_feature_universe"
].astype(bool).all():
    raise ValueError(
        "OOD fold audit 顯示 calibration feature universe 不一致"
    )

if (
    len(
        ood_fold_thresholds
    )
    != tri[
        "lot_number"
    ].nunique()
):
    raise ValueError(
        "OOD fold threshold rows 與 Lot 數不一致"
    )


print(
    "02 / 08 labeled wafer identity：PASS"
)

print(
    "04 candidate → bootstrap lineage：PASS"
)

print(
    "08 prediction error formula：PASS"
)

print(
    "08 Primary → Ablation lineage：PASS"
)

print(
    "08 → 09 → 10 VM error lineage：PASS"
)

print(
    "06 → 10 residual PCA lineage：PASS"
)

print(
    "10 Pure OOD source separation：PASS"
)

print(
    "10 fixed outer feature universe：PASS"
)


02 / 08 labeled wafer identity：PASS
04 candidate → bootstrap lineage：PASS
08 prediction error formula：PASS
08 Primary → Ablation lineage：PASS
08 → 09 → 10 VM error lineage：PASS
06 → 10 residual PCA lineage：PASS
10 Pure OOD source separation：PASS
10 fixed outer feature universe：PASS


### 這個輸出代表什麼？

這格一共做了 **8 項 lineage 檢查，而且全部 PASS**：

1. 02 / 08 labeled wafer identity：PASS
2. 04 candidate → bootstrap lineage：PASS
3. 08 prediction error formula：PASS
4. 08 Primary → Ablation lineage：PASS
5. 08 → 09 → 10 VM error lineage：PASS
6. 06 → 10 residual PCA lineage：PASS
7. 10 Pure OOD source separation：PASS
8. 10 fixed outer feature universe：PASS

這表示 11 現在讀到的 04、08、09、10 artifacts 在 wafer identity、prediction error、PCA lineage 和 OOD 設定上都有對上。

也就是後面 11 整理的數字是同一條資料流程產生的，不是不同版本結果拼在一起。


## 3. Project Headline Metrics：先把整個專案最重要的數字放在一起

這一格只整理最核心的 headline，不做新的模型。

我會同時放：

- Quality 的基本結果
- Sequence structure
- Virtual Metrology performance
- SPC / Residual / OOD 的 review counts

這樣後面每一節再分開解釋。

In [3]:
y = nested_oof[
    "actual_mean_si_etch"
].to_numpy(float)

pred = nested_oof[
    "nested_process_pred"
].to_numpy(float)

vm_rmse = float(
    np.sqrt(
        np.mean(
            (pred - y)
            ** 2
        )
    )
)

vm_mae = float(
    np.mean(
        np.abs(
            pred - y
        )
    )
)

vm_r2 = float(
    r2_score(
        y,
        pred,
    )
)


mean_base = float(
    baseline.loc[
        baseline["model"]
        == "Training-Mean Baseline",
        "rmse",
    ].iloc[0]
)

seq_base = float(
    baseline.loc[
        baseline["model"]
        == "Sequence-Only Linear Baseline",
        "rmse",
    ].iloc[0]
)


ablation_lookup = (
    ablation_metrics
    .set_index(
        "model"
    )
)

process_plus_sequence_rmse = float(
    ablation_lookup.loc[
        "Process + Sequence",
        "rmse",
    ]
)

process_plus_sequence_mae = float(
    ablation_lookup.loc[
        "Process + Sequence",
        "mae",
    ]
)

process_plus_sequence_r2 = float(
    ablation_lookup.loc[
        "Process + Sequence",
        "r2",
    ]
)


inner_mask = (
    radial[
        "radius_region"
    ]
    != ">=75 mm"
)

inner_mean = float(
    np.average(
        radial.loc[
            inner_mask,
            "mean_si_etch",
        ],
        weights=radial.loc[
            inner_mask,
            "n_points",
        ],
    )
)

outer_mean = float(
    radial.loc[
        radial[
            "radius_region"
        ]
        == ">=75 mm",
        "mean_si_etch",
    ].iloc[0]
)


res_count = int(
    residual_anomaly[
        "residual_review_candidate"
    ].sum()
)

ood_count = int(
    cross_lot_ood[
        "cross_lot_ood_candidate"
    ].sum()
)


candidate_overlap = (
    residual_anomaly[
        [
            "experiment_key",
            "residual_review_candidate",
        ]
    ]
    .merge(
        cross_lot_ood[
            [
                "experiment_key",
                "cross_lot_ood_candidate",
            ]
        ],
        on="experiment_key",
        how="inner",
        validate="one_to_one",
    )
)

overlap_count = int(
    (
        candidate_overlap[
            "residual_review_candidate"
        ]
        & candidate_overlap[
            "cross_lot_ood_candidate"
        ]
    ).sum()
)


headline = pd.DataFrame(
    [
        {
            "metric":
                "Process wafers",
            "value":
                len(tri),
            "unit":
                "wafers",
        },
        {
            "metric":
                "Quality-labeled wafers",
            "value":
                len(nested_oof),
            "unit":
                "wafers",
        },
        {
            "metric":
                "Lots",
            "value":
                tri[
                    "lot_number"
                ].nunique(),
            "unit":
                "Lots",
        },
        {
            "metric":
                "Mean Si Etch",
            "value":
                quality[
                    "mean_si_etch"
                ].mean(),
            "unit":
                "µm",
        },
        {
            "metric":
                "Mean CV",
            "value":
                quality[
                    "cv_si_etch_pct"
                ].mean(),
            "unit":
                "%",
        },
        {
            "metric":
                "Outer - inner radial mean",
            "value":
                outer_mean
                - inner_mean,
            "unit":
                "µm",
        },
        {
            "metric":
                "Process-only Nested OOF RMSE",
            "value":
                vm_rmse,
            "unit":
                "µm",
        },
        {
            "metric":
                "Process-only Nested OOF MAE",
            "value":
                vm_mae,
            "unit":
                "µm",
        },
        {
            "metric":
                "Process-only Nested OOF R²",
            "value":
                vm_r2,
            "unit":
                "",
        },
        {
            "metric":
                "Process + Sequence RMSE",
            "value":
                process_plus_sequence_rmse,
            "unit":
                "µm",
        },
        {
            "metric":
                "Process + Sequence MAE",
            "value":
                process_plus_sequence_mae,
            "unit":
                "µm",
        },
        {
            "metric":
                "Process + Sequence R²",
            "value":
                process_plus_sequence_r2,
            "unit":
                "",
        },
        {
            "metric":
                "Sequence-only baseline RMSE",
            "value":
                seq_base,
            "unit":
                "µm",
        },
        {
            "metric":
                "Training-mean baseline RMSE",
            "value":
                mean_base,
            "unit":
                "µm",
        },
        {
            "metric":
                "Residual review candidates",
            "value":
                res_count,
            "unit":
                "wafers",
        },
        {
            "metric":
                "Cross-Lot OOD candidates",
            "value":
                ood_count,
            "unit":
                "wafers",
        },
        {
            "metric":
                "Residual + OOD overlap",
            "value":
                overlap_count,
            "unit":
                "wafers",
        },
    ]
)

display(
    headline.round(5)
)


,metric,value,unit
0,Process wafers,96.00000,wafers
1,Quality-labeled wafers,88.00000,wafers
2,Lots,10.00000,Lots
3,Mean Si Etch,43.66812,µm
4,Mean CV,8.21143,%
5,Outer - inner radial mean,5.03895,µm
6,Process-only Nested OOF RMSE,0.12963,µm
7,Process-only Nested OOF MAE,0.09723,µm
8,Process-only Nested OOF R²,0.89477,
9,Process + Sequence RMSE,0.09540,µm


### 這個輸出代表什麼？

這張 Headline table 把整個專案最重要的數字放在一起。

資料規模：

- Process wafers：**96**
- Quality-labeled wafers：**88**
- Lots：**10**

Quality：

- Mean Si Etch：**43.66812 µm**
- Mean CV：**8.21143%**
- Outer - inner radial mean：**+5.03895 µm**

Virtual Metrology：

**Process-Only**

- RMSE：**0.12963 µm**
- MAE：**0.09723 µm**
- R²：**0.89477**

**Process + Sequence**

- RMSE：**0.09540 µm**
- MAE：**0.07500 µm**
- R²：**0.94301**

Baselines：

- Sequence-only RMSE：**0.22093 µm**
- Training-mean RMSE：**0.40437 µm**

Review candidates：

- Residual review：**3 片**
- Cross-Lot OOD：**5 片**
- 同時被兩種方法標記：**2 片**

從這張表可以直接看到，四種 prediction 設定裡 `Process + Sequence` 的 RMSE 最低，而 Process-only 也明顯低於兩個 baseline。


## 4. Quality 與 Sequence Structure：先確認資料本身有哪些穩定結構

在做 Process–Quality correlation 或模型以前，先把最重要的資料結構整理出來：

- Quality 是否有 Lot 內 run-order trend？
- Process features 是否也跟 wafer sequence 有關？
- 扣掉線性 sequence trend 後，這種 association 有沒有真的下降？
- Main Active Window 的時間範圍是否穩定？

這一節的目的不是證明因果，而是說明為什麼後面的分析不能把 96 片 wafer 當成完全獨立、同分布的資料。

In [4]:
quality_sequence_summary = pd.DataFrame([
    {
        "metric": "Mean Si Etch average",
        "value": float(quality["mean_si_etch"].mean()),
        "note": "88 labeled wafers",
    },
    {
        "metric": "CV average (%)",
        "value": float(quality["cv_si_etch_pct"].mean()),
        "note": "88 labeled wafers",
    },
    {
        "metric": "Lots with negative Mean Si Etch slope",
        "value": int((lot_sequence["mean_si_etch_slope_per_wafer"] < 0).sum()),
        "note": f"out of {len(lot_sequence)} Lots",
    },
    {
        "metric": "Lots with negative CV slope",
        "value": int((lot_sequence["cv_slope_per_wafer"] < 0).sum()),
        "note": f"out of {len(lot_sequence)} Lots",
    },
    {
        "metric": "Main Active Window mean duration (s)",
        "value": float(process_summary["main_active_duration"].mean()),
        "note": "03 data-driven 80% SourceRFLoadPower heuristic",
    },
    {
        "metric": "Main Active Window duration SD (s)",
        "value": float(process_summary["main_active_duration"].std()),
        "note": "96 process wafers",
    },
    {
        "metric": "Median |within-Lot Spearman| before detrending",
        "value": float(feature_sequence_comparison["raw_median_abs_within_lot_spearman"].median()),
        "note": "24 comparable process features",
    },
    {
        "metric": "Median |within-Lot Spearman| after detrending",
        "value": float(feature_sequence_comparison["residual_median_abs_within_lot_spearman"].median()),
        "note": "same 24 process features",
    },
])

display(quality_sequence_summary.round(5))

top_sequence_features = (
    process_sequence_summary
    .sort_values("median_abs_spearman", ascending=False)
    .head(8)
    [["feature", "dominant_direction", "direction_consistency_pct", "median_spearman", "median_abs_spearman"]]
    .copy()
)
top_sequence_features["feature"] = top_sequence_features["feature"].str.replace(
    "Stat3_Etch_MV_", "", regex=False
)

print("\nSequence structure 最明顯的 process features：")
display(top_sequence_features.round(4))

n_decreased = int(
    (
        feature_sequence_comparison["residual_median_abs_within_lot_spearman"]
        < feature_sequence_comparison["raw_median_abs_within_lot_spearman"]
    ).sum()
)
print(
    f"Sequence detrending 後 |Spearman| 下降："
    f"{n_decreased} / {len(feature_sequence_comparison)} features"
)

,metric,value,note
0,Mean Si Etch average,43.66812,88 labeled wafers
1,CV average (%),8.21143,88 labeled wafers
2,Lots with negative Mean Si Etch slope,10.00000,out of 10 Lots
3,Lots with negative CV slope,9.00000,out of 10 Lots
4,Main Active Window mean duration (s),597.54208,03 data-driven 80% SourceRFLoadPower heuristic
5,Main Active Window duration SD (s),2.19824,96 process wafers
6,Median |within-Lot Spearman| before detrending,0.38788,24 comparable process features
7,Median |within-Lot Spearman| after detrending,0.08485,same 24 process features



Sequence structure 最明顯的 process features：


,feature,dominant_direction,direction_consistency_pct,median_spearman,median_abs_spearman
0,PlatenRFTuningCapacitor,Increasing,100.0,0.9576,0.9576
1,SourceRFPeakToPeak,Decreasing,100.0,-0.8061,0.8061
2,ForeLinePressure,Decreasing,100.0,-0.7758,0.7758
3,PlatenRFPeakToPeak,Increasing,100.0,0.7697,0.7697
4,SourceRFLoadPower,Decreasing,100.0,-0.6727,0.6727
8,Gas7Flow,Increasing,80.0,0.5576,0.5576
19,Gas2Flow,Decreasing,60.0,0.0000,0.5152
15,Heater3Temp,Increasing,70.0,0.4182,0.5091


Sequence detrending 後 |Spearman| 下降：24 / 24 features


### 這個輸出代表什麼？

Quality 端的 sequence 結果：

- Mean Si Etch slope 為負：**10 / 10 Lots**
- CV slope 為負：**9 / 10 Lots**

Process 端在 detrending 前後：

- Median `|within-Lot Spearman|`：**0.38788 → 0.08485**
- **24 / 24 features** 在 detrending 後都下降

也就是這 24 個可比較的 Process features，在扣掉 Lot 內 linear wafer-sequence trend 後，和 wafer order 的關聯都變小。

Sequence structure 最明顯的前幾個 features：

- `PlatenRFTuningCapacitor`：median Spearman **+0.9576**，10 Lots 方向一致率 **100%**
- `SourceRFPeakToPeak`：**-0.8061**，一致率 **100%**
- `ForeLinePressure`：**-0.7758**，一致率 **100%**
- `PlatenRFPeakToPeak`：**+0.7697**，一致率 **100%**
- `SourceRFLoadPower`：**-0.6727**，一致率 **100%**

另外 Main Active Window：

- 平均 duration：**597.54208 秒**
- SD：**2.19824 秒**

這格最直接的結果是：Quality 和多個 Process features 都有很明顯的 Lot 內 sequence pattern。


## 5. Process–Quality：扣掉 Lot 內線性 sequence trend 後，還留下哪些候選？

04 的正式 candidate rule 已經在上游完成，11 不再重新篩選。

這裡只整理兩個 target：

- Mean Si Etch
- CV Si Etch

這些候選可以拿來安排 DOE、equipment log review 或 recipe review 的優先順序，但不能直接寫成 causal driver。

In [5]:
def short_sensor(
    series,
):
    return (
        series.astype(str)
        .str.replace(
            "Stat3_Etch_MV_",
            "",
            regex=False,
        )
    )


bootstrap_columns = [
    "feature",
    "bootstrap_n_valid",
    "bootstrap_valid_fraction",
    "bootstrap_ci_low",
    "bootstrap_median",
    "bootstrap_ci_high",
    "bootstrap_excludes_zero",
]


mean_final = (
    mean_candidates
    .merge(
        mean_bootstrap[
            bootstrap_columns
        ],
        on="feature",
        how="left",
        validate="one_to_one",
    )
)

mean_final.insert(
    0,
    "sensor",
    short_sensor(
        mean_final["feature"]
    ),
)

mean_final[
    "target"
] = "Mean Si Etch"


cv_final = (
    cv_candidates
    .merge(
        cv_bootstrap[
            bootstrap_columns
        ],
        on="feature",
        how="left",
        validate="one_to_one",
    )
)

cv_final.insert(
    0,
    "sensor",
    short_sensor(
        cv_final["feature"]
    ),
)

cv_final[
    "target"
] = "CV Si Etch"


process_quality_summary = pd.concat(
    [
        mean_final,
        cv_final,
    ],
    ignore_index=True,
)


print(
    "Mean Si Etch candidates："
)

display(
    mean_final[
        [
            "sensor",
            "sequence_adjusted_pearson",
            "sequence_adjusted_spearman",
            "bootstrap_ci_low",
            "bootstrap_ci_high",
            "bootstrap_excludes_zero",
            "n_valid_lots",
        ]
    ].round(4)
)


print(
    "\nCV Si Etch candidates："
)

display(
    cv_final[
        [
            "sensor",
            "sequence_adjusted_pearson",
            "sequence_adjusted_spearman",
            "bootstrap_ci_low",
            "bootstrap_ci_high",
            "bootstrap_excludes_zero",
            "n_valid_lots",
        ]
    ].round(4)
)


mean_bootstrap_stable = int(
    mean_final[
        "bootstrap_excludes_zero"
    ].sum()
)

cv_bootstrap_stable = int(
    cv_final[
        "bootstrap_excludes_zero"
    ].sum()
)


print(
    "\nBootstrap interval 沒有跨 0："
)

print(
    "Mean Si Etch：",
    f"{mean_bootstrap_stable}/{len(mean_final)}",
)

print(
    "CV Si Etch：",
    f"{cv_bootstrap_stable}/{len(cv_final)}",
)


top_mean = mean_final.loc[
    mean_final[
        "abs_sequence_adjusted_pearson"
    ].idxmax()
]

top_cv = cv_final.loc[
    cv_final[
        "abs_sequence_adjusted_pearson"
    ].idxmax()
]


Mean Si Etch candidates：


,sensor,sequence_adjusted_pearson,sequence_adjusted_spearman,bootstrap_ci_low,bootstrap_ci_high,bootstrap_excludes_zero,n_valid_lots
0,PlatenRFLoadPower,-0.6090,-0.4855,-0.7525,-0.4208,True,10
1,HeliumBPPressure,-0.3999,-0.4042,-0.6149,-0.1729,True,10
2,Gas1Flow,0.3298,0.1753,-0.0248,0.5625,False,10
3,PlatenRFPeakToPeak,-0.3290,-0.3128,-0.5264,-0.1076,True,10
4,Pressure,-0.3079,-0.2382,-0.4096,-0.2019,True,10



CV Si Etch candidates：


,sensor,sequence_adjusted_pearson,sequence_adjusted_spearman,bootstrap_ci_low,bootstrap_ci_high,bootstrap_excludes_zero,n_valid_lots
0,Heater4Temp,0.5513,0.4654,0.1669,0.8201,True,10
1,moriInnerCurrent,-0.5008,-0.5020,-0.7474,-0.2396,True,10
2,Gas7Flow,-0.4201,-0.5653,-0.9078,0.0706,False,10
3,PlatenRFTuningCapacitor,-0.4107,-0.6259,-0.7920,-0.0948,True,10
4,PlatenRFPeakToPeak,-0.4088,-0.5157,-0.7290,-0.0586,True,10
5,Gas2Flow,0.3277,0.2363,-0.1163,0.6803,False,10
6,Heater2Temp,0.3158,0.2005,-0.2426,0.6274,False,10



Bootstrap interval 沒有跨 0：
Mean Si Etch： 4/5
CV Si Etch： 4/7


### 這個輸出代表什麼？

Mean Si Etch 一共有 **5 個 candidates**。

Adjusted Pearson 絕對值最大的是：

- `PlatenRFLoadPower`：**-0.6090**
- Bootstrap interval：**[-0.7525, -0.4208]**
- interval 沒有跨 0

其他 Mean candidates：

- `HeliumBPPressure`：**-0.3999**，CI **[-0.6149, -0.1729]**
- `Gas1Flow`：**+0.3298**，CI **[-0.0248, +0.5625]**
- `PlatenRFPeakToPeak`：**-0.3290**，CI **[-0.5264, -0.1076]**
- `Pressure`：**-0.3079**，CI **[-0.4096, -0.2019]**

所以 Mean candidates 裡有 **4 / 5** 的 bootstrap interval 沒跨 0，只有 `Gas1Flow` 跨 0。

CV Si Etch 一共有 **7 個 candidates**。

Adjusted Pearson 絕對值最大的是：

- `Heater4Temp`：**+0.5513**
- Bootstrap interval：**[+0.1669, +0.8201]**

另外：

- `moriInnerCurrent`：**-0.5008**，CI **[-0.7474, -0.2396]**
- `Gas7Flow`：**-0.4201**，CI **[-0.9078, +0.0706]**
- `PlatenRFTuningCapacitor`：**-0.4107**，CI **[-0.7920, -0.0948]**
- `PlatenRFPeakToPeak`：**-0.4088**，CI **[-0.7290, -0.0586]**
- `Gas2Flow`：**+0.3277**，CI **[-0.1163, +0.6803]**
- `Heater2Temp`：**+0.3158**，CI **[-0.2426, +0.6274]**

CV candidates 裡也是 **4 / 7** 的 interval 沒跨 0。

所以這格可以看出，雖然這些 features 都通過原本 candidate rule，但 bootstrap uncertainty 並不完全一樣。


## 6. SPC 與 Multivariate Structure：哪些 wafer / feature 值得先回查？

05 是 retrospective Phase-I SPC；06 則用 PCA 看多變量結構。

這裡要特別分清楚：

- I-chart flag：某片 wafer 的 residual level 比較極端。
- MR flag：前一片到這一片的變化比較大。
- PCA：只是幫忙描述多變量 variation，不是製程物理模型。

所以這些訊號適合當 investigation trigger，不是 production defect label。

In [6]:
spc_summary = pd.DataFrame([
    {
        "metric": "I-chart feature-wafer flags",
        "value": int(spc_wafer["n_flagged_features"].sum()),
    },
    {
        "metric": "Unique wafers with I-chart flag",
        "value": int((spc_wafer["n_flagged_features"] > 0).sum()),
    },
    {
        "metric": "MR feature-wafer flags",
        "value": int(spc_wafer["n_mr_flagged_features"].sum()),
    },
    {
        "metric": "Unique wafers with MR flag",
        "value": int((spc_wafer["n_mr_flagged_features"] > 0).sum()),
    },
    {
        "metric": "Raw PCA variance explained by PC1-PC3",
        "value": float(raw_pca_var["explained_variance_ratio"].head(3).sum()),
    },
    {
        "metric": "Raw PCA components for >=90%",
        "value": int(raw_pca_var.loc[raw_pca_var["cumulative_variance_ratio"] >= 0.90, "component"].iloc[0]),
    },
    {
        "metric": "Residual PCA components for >=90%",
        "value": int(residual_pca_var.loc[residual_pca_var["cumulative_variance_ratio"] >= 0.90, "component"].iloc[0]),
    },
])

display(spc_summary.round(5))

print("\nI-chart violations 最多的 features：")
top_spc_features = (
    spc_feature.sort_values("n_i_violations", ascending=False)
    .head(8)
    [["feature", "n_i_violations", "i_violation_rate_pct", "n_mr_violations", "mr_violation_rate_pct"]]
    .copy()
)
top_spc_features["feature"] = top_spc_features["feature"].str.replace(
    "Stat3_Etch_MV_", "", regex=False
)
display(top_spc_features.round(4))

,metric,value
0,I-chart feature-wafer flags,46.00000
1,Unique wafers with I-chart flag,19.00000
2,MR feature-wafer flags,47.00000
3,Unique wafers with MR flag,23.00000
4,Raw PCA variance explained by PC1-PC3,0.48884
5,Raw PCA components for >=90%,12.00000
6,Residual PCA components for >=90%,12.00000



I-chart violations 最多的 features：


,feature,n_i_violations,i_violation_rate_pct,n_mr_violations,mr_violation_rate_pct
0,Gas7Flow,8,8.3333,4,4.6512
1,PlatenRFTuningCapacitor,7,7.2917,0,0.0000
2,Heater4Temp,5,5.2083,2,2.3256
3,Gas1Flow,5,5.2083,8,9.3023
4,Heater2Temp,3,3.1250,2,2.3256
5,SourceRFLoadPower,3,3.1250,5,5.8140
6,SourceRF2TuningCapacitor,3,5.0000,4,7.4074
7,Gas2Flow,2,2.0833,2,2.3256


### 這個輸出代表什麼？

Phase-I SPC：

- I-chart feature-wafer flags：**46**
- 有 I-chart flag 的 unique wafers：**19**
- MR feature-wafer flags：**47**
- 有 MR flag 的 unique wafers：**23**

I-chart violations 最多的 feature 是：

- `Gas7Flow`：**8 次**，violation rate **8.3333%**

接下來是：

- `PlatenRFTuningCapacitor`：**7 次**
- `Heater4Temp`：**5 次**
- `Gas1Flow`：**5 次**

MR violations 裡，表格中最多的是：

- `Gas1Flow`：**8 次**，MR violation rate **9.3023%**

PCA 部分：

- Raw PCA 前 3 PCs 累積解釋：**48.884%**
- Raw PCA 到 90% 需要：**12 PCs**
- Residual PCA 到 90% 也需要：**12 PCs**

所以這格的結果顯示，SPC flags 分散在多個 feature / wafer 上，而 PCA 也不是只靠前 1～3 個 component 就能涵蓋大部分 variation。


## 7. Virtual Metrology：正式成績只看 Nested Leave-One-Lot-Out

08 的 full-data final model 是 prototype configuration；真正可以拿來談 generalization 的成績，仍然是 Nested Leave-One-Lot-Out OOF。

這裡同時保留兩個簡單 baseline：

- Training-Mean baseline
- Sequence-Only Linear baseline

這樣可以回答 process-feature model 到底有沒有比很簡單的做法更好。

In [7]:
print(
    "Full-data prototype configuration："
)

print(
    final_config[
        "feature_set"
    ],
    "+",
    final_config[
        "model"
    ],
    final_config[
        "params"
    ],
)

print(
    "Input predictors / variance filter 後保留：",
    final_config[
        "n_input_predictors"
    ],
    "/",
    final_config[
        "n_predictors_after_variance_filter"
    ],
)


vm_summary = pd.DataFrame(
    [
        {
            "metric":
                "Training-Mean RMSE (µm)",
            "value":
                mean_base,
        },
        {
            "metric":
                "Sequence-Only RMSE (µm)",
            "value":
                seq_base,
        },
        {
            "metric":
                "Process-Only RMSE (µm)",
            "value":
                vm_rmse,
        },
        {
            "metric":
                "Process-Only MAE (µm)",
            "value":
                vm_mae,
        },
        {
            "metric":
                "Process-Only R²",
            "value":
                vm_r2,
        },
        {
            "metric":
                "Process + Sequence RMSE (µm)",
            "value":
                process_plus_sequence_rmse,
        },
        {
            "metric":
                "Process + Sequence MAE (µm)",
            "value":
                process_plus_sequence_mae,
        },
        {
            "metric":
                "Process + Sequence R²",
            "value":
                process_plus_sequence_r2,
        },
        {
            "metric":
                "Process + Sequence RMSE reduction vs Process-Only",
            "value":
                (
                    vm_rmse
                    - process_plus_sequence_rmse
                )
                / vm_rmse,
        },
    ]
)

display(
    vm_summary.round(5)
)


ablation_lot_comparison = (
    outer[
        [
            "test_lot",
            "n_test",
            "outer_rmse",
            "mean_baseline_rmse",
            "sequence_baseline_rmse",
        ]
    ]
    .rename(
        columns={
            "outer_rmse":
                "process_only_rmse",
        }
    )
    .merge(
        ablation_outer[
            [
                "test_lot",
                "feature_set",
                "model",
                "outer_rmse",
            ]
        ].rename(
            columns={
                "feature_set":
                    "process_plus_sequence_feature_set",
                "model":
                    "process_plus_sequence_model",
                "outer_rmse":
                    "process_plus_sequence_rmse",
            }
        ),
        on="test_lot",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        "test_lot"
    )
    .reset_index(
        drop=True
    )
)

ablation_lot_comparison[
    "rmse_change_vs_process_only"
] = (
    ablation_lot_comparison[
        "process_plus_sequence_rmse"
    ]
    - ablation_lot_comparison[
        "process_only_rmse"
    ]
)


print(
    "\nPer-Lot Process-Only vs Process + Sequence："
)

display(
    ablation_lot_comparison.round(5)
)


n_ablation_better = int(
    (
        ablation_lot_comparison[
            "process_plus_sequence_rmse"
        ]
        <
        ablation_lot_comparison[
            "process_only_rmse"
        ]
    ).sum()
)

print(
    "\nProcess + Sequence RMSE 較低：",
    f"{n_ablation_better}/10 Lots",
)


print(
    "\nPrimary Process-Only outer-fold selected feature sets："
)

display(
    outer[
        "feature_set"
    ]
    .value_counts()
    .rename_axis(
        "feature_set"
    )
    .reset_index(
        name="folds"
    )
)


print(
    "\nPrimary Process-Only outer-fold selected model families："
)

display(
    outer[
        "model"
    ]
    .value_counts()
    .rename_axis(
        "model"
    )
    .reset_index(
        name="folds"
    )
)


print(
    "\nProcess + Sequence outer-fold selected feature sets："
)

display(
    ablation_outer[
        "feature_set"
    ]
    .value_counts()
    .rename_axis(
        "feature_set"
    )
    .reset_index(
        name="folds"
    )
)


print(
    "\nProcess + Sequence outer-fold selected model families："
)

display(
    ablation_outer[
        "model"
    ]
    .value_counts()
    .rename_axis(
        "model"
    )
    .reset_index(
        name="folds"
    )
)


Full-data prototype configuration：
Extended + ElasticNet {'alpha': 0.01, 'l1_ratio': 0.1}
Input predictors / variance filter 後保留： 109 / 98


,metric,value
0,Training-Mean RMSE (µm),0.40437
1,Sequence-Only RMSE (µm),0.22093
2,Process-Only RMSE (µm),0.12963
3,Process-Only MAE (µm),0.09723
4,Process-Only R²,0.89477
5,Process + Sequence RMSE (µm),0.09540
6,Process + Sequence MAE (µm),0.07500
7,Process + Sequence R²,0.94301
8,Process + Sequence RMSE reduction vs Process-Only,0.26406



Per-Lot Process-Only vs Process + Sequence：


,test_lot,n_test,process_only_rmse,mean_baseline_rmse,sequence_baseline_rmse,process_plus_sequence_feature_set,process_plus_sequence_model,process_plus_sequence_rmse,rmse_change_vs_process_only
0,1,9,0.06516,0.40249,0.16795,Base,ElasticNet,0.05377,-0.01139
1,2,10,0.10996,0.45151,0.20434,Base,ElasticNet,0.08010,-0.02986
2,3,10,0.14304,0.39116,0.21609,Base,ElasticNet,0.11542,-0.02762
3,4,10,0.11432,0.50858,0.26757,Base,ElasticNet,0.08532,-0.02899
4,5,10,0.13771,0.42123,0.23254,Base,ElasticNet,0.11446,-0.02325
5,6,10,0.11752,0.41429,0.24175,Base,ElasticNet,0.09393,-0.02359
6,7,6,0.11837,0.23659,0.20841,Base,ElasticNet,0.13705,0.01868
7,8,10,0.22384,0.20788,0.23629,Base,ElasticNet,0.10707,-0.11677
8,9,9,0.06980,0.42305,0.21132,Base,ElasticNet,0.05778,-0.01202
9,10,4,0.08315,0.47620,0.15400,Base,ElasticNet,0.08383,0.00067



Process + Sequence RMSE 較低： 8/10 Lots

Primary Process-Only outer-fold selected feature sets：


...

,model,folds
0,ElasticNet,10



Process + Sequence outer-fold selected feature sets：


,feature_set,folds
0,Base,10



Process + Sequence outer-fold selected model families：


,model,folds
0,ElasticNet,10


### 這個輸出代表什麼？

Final full-data prototype configuration 是：

- **Extended + ElasticNet**
- `alpha = 0.01`
- `l1_ratio = 0.1`
- Input predictors：**109**
- Variance filter 後保留：**98**

四種 prediction 設定的整體 RMSE：

- Training-Mean：**0.40437**
- Sequence-Only：**0.22093**
- Process-Only：**0.12963**
- Process + Sequence：**0.09540**

Process + Sequence 相比 Process-Only：

- RMSE：**0.12963 → 0.09540**
- Output 算出的 reduction：**0.26406**，也就是約 **26.4%**
- MAE：**0.09723 → 0.07500**
- R²：**0.89477 → 0.94301**

分 Lot 比較時，Process + Sequence RMSE 較低的是 **8 / 10 Lots**。

改善最大的 Lot 是 **Lot 8**：

- Process-Only：**0.22384**
- Process + Sequence：**0.10707**
- 差：**-0.11677**

有兩個 Lot 沒有變好：

- Lot 7：**0.11837 → 0.13705**，增加 **0.01868**
- Lot 10：**0.08315 → 0.08383**，增加 **0.00067**

Model selection：

**Primary Process-Only**
- ElasticNet：**10 / 10 folds**
- Extended：**6 folds**
- Base：**4 folds**

**Process + Sequence**
- ElasticNet：**10 / 10 folds**
- Base：**10 / 10 folds**

所以這格最明確的結果是：加入 `wafer_number` 後，整體 prediction 指標變好，而且 10 個 held-out Lots 裡有 8 個 RMSE 下降；但 Lot 7 和 Lot 10 沒有下降。


## 8. Explainability：模型最依賴哪些 sensor groups？

09 有兩種互補的 model-dependent evidence：

1. Nested outer models 的 grouped permutation sensitivity。
2. Final ElasticNet 的 standardized coefficient aggregation。

如果兩邊都指向同一個 sensor group，可以說模型對這組資訊的依賴比較一致；但仍然不能直接把它叫作 causal process driver。

In [8]:
top_perm = (
    sensor_perm
    .sort_values("mean_delta_rmse", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

print("Nested grouped permutation sensitivity：")
display(top_perm.round(5))

top_coef = (
    sensor_coef
    .sort_values("abs_coefficient_sum", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

print("\nFinal ElasticNet sensor coefficient aggregation：")
display(top_coef.round(5))

top_perm_sensor = str(top_perm.iloc[0]["sensor"])
top_coef_sensor = str(top_coef.iloc[0]["sensor_short"])
print("\nTop sensor agrees across both views：", top_perm_sensor == top_coef_sensor)

Nested grouped permutation sensitivity：


,sensor,n_repeats,mean_delta_rmse,std_delta_rmse,positive_repeat_fraction
0,PlatenRFTuningCapacitor,30,0.24996,0.02185,1.00000
1,MainActiveWindow,30,0.05900,0.01242,1.00000
2,ForeLinePressure,30,0.03235,0.00597,1.00000
3,SourceRFReflectedPower,30,0.02489,0.00461,1.00000
4,PlatenRFReflectedPower,30,0.01596,0.00366,1.00000
5,SourceRFLoadPower,30,0.01398,0.00440,1.00000
6,SourceRFPeakToPeak,30,0.00924,0.00344,1.00000
7,PlatenRFLoadCapacitor,30,0.00333,0.00187,0.96667
8,PlatenRFLoadPower,30,0.00200,0.00094,0.96667
9,Gas2Flow,30,0.00151,0.00131,0.83333



Final ElasticNet sensor coefficient aggregation：


,sensor_short,n_features,n_nonzero,abs_coefficient_sum,max_abs_coefficient
0,PlatenRFTuningCapacitor,4,3,0.26939,0.15959
1,PlatenRFReflectedPower,4,3,0.11761,0.09083
2,ForeLinePressure,4,4,0.10781,0.07588
3,MainActiveWindow,1,1,0.09182,0.09182
4,SourceRFReflectedPower,4,2,0.07620,0.05684
5,Heater2Temp,4,3,0.07293,0.05977
6,SourceRFPeakToPeak,4,2,0.05505,0.04177
7,PlatenRFPeakToPeak,4,3,0.05198,0.02665
8,SourceRFLoadPower,4,2,0.05103,0.04794
9,Gas7Flow,4,3,0.04582,0.02355



Top sensor agrees across both views： True


### 這個輸出代表什麼？

Grouped permutation sensitivity 排名第一的是：

- `PlatenRFTuningCapacitor`
- mean ΔRMSE：**+0.24996**
- std：**0.02185**
- 30 次 permutation 中 positive fraction：**1.0**

第二名 `MainActiveWindow` 的 mean ΔRMSE 是 **+0.05900**，和第一名差距很大。

Final ElasticNet coefficient aggregation 第一名也同樣是：

- `PlatenRFTuningCapacitor`
- `abs_coefficient_sum = 0.26939`
- 4 個 engineered features 裡有 **3 個 non-zero**
- 最大單一 absolute coefficient：**0.15959**

最後輸出：

`Top sensor agrees across both views：True`

也就是 grouped permutation 和 coefficient aggregation 這兩種結果的第一名是同一個 sensor group。


## 9. Residual Anomaly、Pure Cross-Lot OOD 與 Model Error：三件事分開看

10 已經把這幾個概念分開：

- **Residual anomaly**：看扣掉 Lot-specific linear sequence trend 後，還有沒有少見 residual pattern。
- **Pure Cross-Lot OOD**：只用 07 process engineered features，看 held-out Lot 相對其他 Lots 有多陌生。
- **VM-informed diagnostic**：可以用 09 supervised model sensitivity 做 post-hoc model context，但它不參與 Pure OOD candidate flag。

11 只讀 10 的正式輸出，不重新建立 detector。

In [9]:
res_candidates = (
    residual_anomaly.loc[
        residual_anomaly["residual_review_candidate"],
        ["experiment_key", "lot_number", "wafer_number", "residual_anomaly_consensus", "residual_driver_feature"],
    ]
    .sort_values(["lot_number", "wafer_number"])
    .reset_index(drop=True)
)

ood_candidates = (
    cross_lot_ood.loc[
        cross_lot_ood["cross_lot_ood_candidate"],
        ["experiment_key", "lot_number", "wafer_number", "cross_lot_ood_consensus", "ood_driver_feature"],
    ]
    .sort_values(["lot_number", "wafer_number"])
    .reset_index(drop=True)
)

overlap_keys = sorted(
    set(res_candidates["experiment_key"])
    & set(ood_candidates["experiment_key"])
)

print("Residual review candidates：", len(res_candidates))
display(res_candidates)

print("Cross-Lot OOD candidates：", len(ood_candidates))
display(ood_candidates)

print("Residual + OOD overlap：", len(overlap_keys), overlap_keys)

print("\nProcess score vs |OOF error|：")
display(association.round(5))

print("\nPriority cases：")
display(priority_cases.round(5))

strong_assoc = association.loc[
    association["spearman_rho_with_abs_oof_error"].abs().idxmax()
]

Residual review candidates： 3


,experiment_key,lot_number,wafer_number,residual_anomaly_consensus,residual_driver_feature
0,2024-08-01_04,6,4,2,Stat3_Etch_MV_Heater3Temp
1,2024-08-07_08,8,8,2,Stat3_Etch_MV_Heater4Temp
2,2024-08-07_09,8,9,2,Stat3_Etch_MV_Heater2Temp


Cross-Lot OOD candidates： 5


,experiment_key,lot_number,wafer_number,cross_lot_ood_consensus,ood_driver_feature
0,2024-07-09_01,3,1,3,std__Gas7Flow
1,2024-07-19_01,5,1,2,std__PlatenRFTuningCapacitor
2,2024-08-07_08,8,8,3,late_minus_early__SourceRFPeakToPeak
3,2024-08-07_09,8,9,3,late_minus_early__Heater2Temp
4,2024-08-07_10,8,10,2,late_minus_early__Heater2Temp


Residual + OOD overlap： 2 ['2024-08-07_08', '2024-08-07_09']

Process score vs |OOF error|：


,metric,spearman_rho_with_abs_oof_error,abs_spearman_rho,lot_bootstrap_ci_low,lot_bootstrap_ci_high,bootstrap_valid_reps,n_labeled_wafers,n_lots
0,ood_pca_recon_rmse,0.30948,0.30948,0.00848,0.60309,2000,88,10
1,ood_mahalanobis,0.23648,0.23648,-0.10785,0.59123,2000,88,10
2,residual_mahalanobis,0.11958,0.11958,-0.10355,0.32406,2000,88,10
3,residual_pca_recon_rmse,0.03013,0.03013,-0.18959,0.25073,2000,88,10
4,vm_informed_max_abs_z,0.01040,0.01040,-0.32242,0.34775,2000,88,10



Priority cases：


,case_type,experiment_key,lot_number,wafer_number,residual_anomaly_consensus,cross_lot_ood_consensus,n_flagged_features,n_mr_flagged_features,cv_si_etch_pct,abs_nested_error,vm_informed_max_abs_z,residual_driver_feature,ood_driver_feature,selection_rule
0,A_Process+Model,2024-08-07_10,8,10,1,2,3,5,8.16625,0.57555,1.86279,Stat3_Etch_MV_Heater4Temp,late_minus_early__Heater2Temp,有 process-review evidence 的 labeled wafers 中，|...
1,B_Quality_Contrast,2024-08-01_07,6,7,0,0,0,0,8.94692,0.18857,2.60485,Stat3_Etch_MV_Pressure,robust_width__PlatenRFReflectedPower,Quality CV 位於高端，而且目前 process-review channels 都...


### 這個輸出代表什麼？

Residual review candidates 一共有 **3 片**：

- `2024-08-01_04`，Lot 6 Wafer 4
- `2024-08-07_08`，Lot 8 Wafer 8
- `2024-08-07_09`，Lot 8 Wafer 9

Cross-Lot OOD candidates 一共有 **5 片**：

- `2024-07-09_01`
- `2024-07-19_01`
- `2024-08-07_08`
- `2024-08-07_09`
- `2024-08-07_10`

兩種方法重疊的只有 **2 片**：

- `2024-08-07_08`
- `2024-08-07_09`

Process score 和 `|OOF error|` 的 Spearman 裡，最高的是：

- `ood_pca_recon_rmse`
- rho = **0.30948**
- 95% bootstrap interval = **[0.00848, 0.60309]**

其他幾個 rho 都更低，而且它們的 interval 都跨 0。

Priority Case A：

- `2024-08-07_10`
- OOD consensus = **2**
- I-chart flagged features = **3**
- MR flagged features = **5**
- `abs_nested_error = 0.57555`

Priority Case B：

- `2024-08-01_07`
- CV = **8.94692**
- Residual consensus = **0**
- OOD consensus = **0**
- I-chart flags = **0**
- MR flags = **0**

所以這格直接顯示：Residual、OOD、SPC、Quality CV 和 VM error 並不會固定一起出現。


## 10. Final Findings Table：把整個專案真正能說的結論濃縮成一張表

這張表不是再做一次統計，而是把 02～10 的主要發現收斂成 Final Findings。

每一列都保留四件事：

- Finding
- Supporting evidence
- Correct interpretation
- Limitation

這樣面試時不會只背漂亮數字，也能清楚說明自己知道哪些結論不能下太大。

In [10]:
top_mean = mean_final.loc[
    mean_final[
        "abs_sequence_adjusted_pearson"
    ].idxmax()
]

top_cv = cv_final.loc[
    cv_final[
        "abs_sequence_adjusted_pearson"
    ].idxmax()
]

top_spc_row = (
    spc_feature
    .sort_values(
        "n_i_violations",
        ascending=False,
    )
    .iloc[0]
)

lot8 = (
    lot_summary.loc[
        lot_summary[
            "lot_number"
        ]
        == 8
    ]
    .iloc[0]
)

lot8_ablation = (
    ablation_lot_comparison.loc[
        ablation_lot_comparison[
            "test_lot"
        ]
        == 8
    ]
    .iloc[0]
)


final_findings = pd.DataFrame(
    [
        {
            "finding":
                "Wafer quality 有 radial spatial structure",

            "supporting_evidence":
                (
                    f"Outer mean={outer_mean:.3f} µm；"
                    f"inner pooled mean={inner_mean:.3f} µm；"
                    f"差={outer_mean-inner_mean:+.3f} µm"
                ),

            "correct_interpretation":
                "外圈和內部量測區的平均 Si Etch level 不一樣。",

            "limitation":
                "這是 88 片量測 wafer 的 observational spatial summary。",
        },

        {
            "finding":
                "Lot 內 wafer sequence structure 很明顯",

            "supporting_evidence":
                (
                    "Mean Si Etch slope<0："
                    f"{int((lot_sequence['mean_si_etch_slope_per_wafer']<0).sum())}/10 Lots；"
                    "24/24 process features detrending 後 median |Spearman| 下降"
                ),

            "correct_interpretation":
                "Quality 和多個 Process features 都會隨 Lot 內 wafer order 系統性變化。",

            "limitation":
                "目前主要處理 Lot-specific linear trend。",
        },

        {
            "finding":
                "Sequence-adjusted Process–Quality candidates 有不同的 bootstrap uncertainty",

            "supporting_evidence":
                (
                    f"Mean candidates={len(mean_final)}，"
                    f"{mean_bootstrap_stable}/{len(mean_final)} interval 不跨 0；"
                    f"CV candidates={len(cv_final)}，"
                    f"{cv_bootstrap_stable}/{len(cv_final)} interval 不跨 0"
                ),

            "correct_interpretation":
                "候選 feature 的 adjusted correlation 不一定在 Lot-level bootstrap 下都維持同一方向。",

            "limitation":
                "目前只有 10 Lots，而且 bootstrap 是 uncertainty summary，不是 causal test。",
        },

        {
            "finding":
                "Phase-I SPC 找到 feature-level excursions",

            "supporting_evidence":
                (
                    f"I-chart flags={int(spc_wafer['n_flagged_features'].sum())} events / "
                    f"{int((spc_wafer['n_flagged_features']>0).sum())} wafers；"
                    f"最多={str(top_spc_row['feature']).replace('Stat3_Etch_MV_','')}"
                ),

            "correct_interpretation":
                "有些 wafer / feature 在目前 retrospective Phase-I limits 下被標記。",

            "limitation":
                "這些不是 production Phase-II control limits。",
        },

        {
            "finding":
                "Process + Sequence ablation 的 held-out-Lot error 低於 Process-only",

            "supporting_evidence":
                (
                    f"Process-only RMSE={vm_rmse:.4f}；"
                    f"Process+Sequence RMSE={process_plus_sequence_rmse:.4f}；"
                    f"{n_ablation_better}/10 Lots RMSE 較低"
                ),

            "correct_interpretation":
                "把 wafer_number 加進 process predictors 後，這次 nested held-out-Lot evaluation 的整體 RMSE 更低。",

            "limitation":
                "這是 predictive ablation，不是 wafer sequence 或 process feature 的 causal effect。",
        },

        {
            "finding":
                "不同 Lots 的 Primary Process-only VM error 差距很大",

            "supporting_evidence":
                (
                    f"Lot 8 RMSE={lot8['vm_rmse']:.4f} µm；"
                    f"Lot 1 RMSE={float(lot_summary.loc[lot_summary['lot_number']==1,'vm_rmse'].iloc[0]):.4f} µm"
                ),

            "correct_interpretation":
                "Primary Process-only model 在不同 held-out Lots 的 prediction error 不一樣。",

            "limitation":
                "Leave-One-Lot-Out 是 internal validation，不是 future chronological test。",
        },

        {
            "finding":
                "模型對少數 sensor groups 的 predictive dependence 較明顯",

            "supporting_evidence":
                (
                    f"Top permutation sensor={top_perm.iloc[0]['sensor']}；"
                    f"mean ΔRMSE={top_perm.iloc[0]['mean_delta_rmse']:.4f} µm"
                ),

            "correct_interpretation":
                "Permutation 後 RMSE 增加較多的 sensor group，對目前 prediction 影響較大。",

            "limitation":
                "Predictive importance 不等於 causal importance。",
        },

        {
            "finding":
                "Residual anomaly 和 Pure Cross-Lot OOD 標記不完全相同",

            "supporting_evidence":
                (
                    f"Residual candidates={len(res_candidates)}；"
                    f"OOD candidates={len(ood_candidates)}；"
                    f"overlap={len(overlap_keys)}"
                ),

            "correct_interpretation":
                "兩種方法實際標到的 wafers 只有部分重疊。",

            "limitation":
                "兩種 threshold 和 score 的定義不同，不能直接互換。",
        },

        {
            "finding":
                "Process unusualness 和 Primary Process-only model error 只有部分同步",

            "supporting_evidence":
                (
                    f"最大 |rho|={abs(strong_assoc['spearman_rho_with_abs_oof_error']):.3f}；"
                    f"95% interval=[{strong_assoc['lot_bootstrap_ci_low']:.3f}, "
                    f"{strong_assoc['lot_bootstrap_ci_high']:.3f}]"
                ),

            "correct_interpretation":
                "目前 process unusualness score 和 |OOF error| 的 rank association 不高。",

            "limitation":
                "目前只有 10 Lots，interval 仍然很寬。",
        },

        {
            "finding":
                "Lot 8 在 Primary VM 與多個 review channels 的數值都比較突出",

            "supporting_evidence":
                (
                    f"Primary VM RMSE={lot8['vm_rmse']:.4f} µm；"
                    f"Process+Sequence RMSE={lot8_ablation['process_plus_sequence_rmse']:.4f} µm；"
                    f"OOD={int(lot8['cross_lot_ood_count'])}；"
                    f"residual={int(lot8['residual_review_count'])}；"
                    f"SPC-I wafers={int(lot8['spc_i_wafer_count'])}"
                ),

            "correct_interpretation":
                "Lot 8 的 Primary Process-only RMSE、OOD、residual 與 SPC-I count 都高於多數 Lots；加入 sequence 後 RMSE 明顯下降。",

            "limitation":
                "這些數字不能直接轉成 defect probability。",
        },
    ]
)

display(
    final_findings
)


,finding,supporting_evidence,correct_interpretation,limitation
0,Wafer quality 有 radial spatial structure,Outer mean=46.216 µm；inner pooled mean=41.177 ...,外圈和內部量測區的平均 Si Etch level 不一樣。,這是 88 片量測 wafer 的 observational spatial summary。
1,Lot 內 wafer sequence structure 很明顯,Mean Si Etch slope<0：10/10 Lots；24/24 process ...,Quality 和多個 Process features 都會隨 Lot 內 wafer o...,目前主要處理 Lot-specific linear trend。
2,Sequence-adjusted Process–Quality candidates 有...,Mean candidates=5，4/5 interval 不跨 0；CV candida...,候選 feature 的 adjusted correlation 不一定在 Lot-lev...,目前只有 10 Lots，而且 bootstrap 是 uncertainty summar...
3,Phase-I SPC 找到 feature-level excursions,I-chart flags=46 events / 19 wafers；最多=Gas7Flow,有些 wafer / feature 在目前 retrospective Phase-I l...,這些不是 production Phase-II control limits。
4,Process + Sequence ablation 的 held-out-Lot err...,Process-only RMSE=0.1296；Process+Sequence RMSE...,把 wafer_number 加進 process predictors 後，這次 nest...,這是 predictive ablation，不是 wafer sequence 或 pro...
5,不同 Lots 的 Primary Process-only VM error 差距很大,Lot 8 RMSE=0.2238 µm；Lot 1 RMSE=0.0652 µm,Primary Process-only model 在不同 held-out Lots 的...,Leave-One-Lot-Out 是 internal validation，不是 fut...
6,模型對少數 sensor groups 的 predictive dependence 較明顯,Top permutation sensor=PlatenRFTuningCapacitor...,Permutation 後 RMSE 增加較多的 sensor group，對目前 pred...,Predictive importance 不等於 causal importance。
7,Residual anomaly 和 Pure Cross-Lot OOD 標記不完全相同,Residual candidates=3；OOD candidates=5；overlap=2,兩種方法實際標到的 wafers 只有部分重疊。,兩種 threshold 和 score 的定義不同，不能直接互換。
8,Process unusualness 和 Primary Process-only mod...,"最大 |rho|=0.309；95% interval=[0.008, 0.603]",目前 process unusualness score 和 |OOF error| 的 r...,目前只有 10 Lots，interval 仍然很寬。
9,Lot 8 在 Primary VM 與多個 review channels 的數值都比較突出,Primary VM RMSE=0.2238 µm；Process+Sequence RMS...,Lot 8 的 Primary Process-only RMSE、OOD、residual...,這些數字不能直接轉成 defect probability。


### 這個輸出代表什麼？

Final Findings table 一共整理出 **10 個 finding**。

這 10 個結果分別涵蓋：

1. Radial spatial structure
2. Wafer sequence structure
3. Process–Quality candidate 的 bootstrap uncertainty
4. Phase-I SPC flags
5. Process + Sequence ablation
6. 不同 Lots 的 Primary Process-only VM error
7. Sensor predictive dependence
8. Residual anomaly 和 Cross-Lot OOD overlap
9. Process unusualness 和 VM error 的 association
10. Lot 8 的多個 review signals

這張表沒有重新計算新的模型或 threshold，而是把前面 Notebook 已經跑出的 evidence 放成：

- `supporting_evidence`
- `correct_interpretation`
- `limitation`

例如 VM 這列實際使用：

- Process-only RMSE = **0.1296**
- Process + Sequence RMSE = **0.0954**
- **8 / 10 Lots** 的 RMSE 較低

所以這張表現在已經和最新 04 Bootstrap 及 08 Ablation 結果對上。


## 11. Lot-Level Final Summary：最後看哪些 Lots 同時出現多個 review signals

這裡只把不同 channel 並排，不做加權 risk score。

因為：

- VM RMSE 是 prediction difficulty。
- OOD count 是 distribution extrapolation。
- SPC 是 retrospective statistical excursion。
- Quality CV 是 metrology variation。

它們的單位和意義都不同，直接相加反而會失真。

In [11]:
lot_final = (
    lot_summary
    .sort_values(
        "lot_number"
    )
    .reset_index(
        drop=True
    )
    .merge(
        ablation_lot_comparison[
            [
                "test_lot",
                "process_plus_sequence_rmse",
                "rmse_change_vs_process_only",
            ]
        ].rename(
            columns={
                "test_lot":
                    "lot_number",
            }
        ),
        on="lot_number",
        how="left",
        validate="one_to_one",
    )
)

display(
    lot_final.round(5)
)


lot_high_rmse = (
    lot_final.loc[
        lot_final[
            "vm_rmse"
        ].idxmax()
    ]
)

lot_low_rmse = (
    lot_final.loc[
        lot_final[
            "vm_rmse"
        ].idxmin()
    ]
)

max_ood_count = int(
    lot_final[
        "cross_lot_ood_count"
    ].max()
)

lot_most_ood = (
    lot_final.loc[
        lot_final[
            "cross_lot_ood_count"
        ]
        == max_ood_count,
        "lot_number",
    ]
    .astype(int)
    .tolist()
)


print(
    "Primary Process-only VM RMSE highest：",
    f"Lot {int(lot_high_rmse['lot_number'])} = {lot_high_rmse['vm_rmse']:.5f} µm",
)

print(
    "Primary Process-only VM RMSE lowest：",
    f"Lot {int(lot_low_rmse['lot_number'])} = {lot_low_rmse['vm_rmse']:.5f} µm",
)

print(
    "Cross-Lot OOD candidates most：",
    ", ".join(
        f"Lot {lot}"
        for lot in lot_most_ood
    ),
    f"({max_ood_count} wafers)",
)

lot8_row = (
    lot_final.loc[
        lot_final[
            "lot_number"
        ]
        == 8
    ]
    .iloc[0]
)

print(
    "Lot 8 Process + Sequence RMSE：",
    f"{lot8_row['process_plus_sequence_rmse']:.5f} µm",
)


,lot_number,n_process_wafers,residual_review_count,mean_residual_mahalanobis,cross_lot_ood_count,mean_ood_mahalanobis,spc_i_wafer_count,spc_mr_wafer_count,n_quality_wafers,mean_quality_cv,high_quality_cv_count,vm_rmse,vm_max_abs_error,process_plus_sequence_rmse,rmse_change_vs_process_only
0,1,10,0,4.20929,0,10.79555,1,2,9,8.07826,0,0.06516,0.11583,0.05377,-0.01139
1,2,10,0,4.06710,0,9.67794,1,1,10,8.02445,1,0.10996,0.16586,0.08010,-0.02986
2,3,10,0,4.22266,1,9.67501,2,4,10,8.22720,0,0.14304,0.22756,0.11542,-0.02762
3,4,10,0,4.32545,0,9.46521,3,1,10,8.21564,1,0.11432,0.21846,0.08532,-0.02899
4,5,10,0,4.13099,1,10.53817,3,3,10,8.14189,1,0.13771,0.27007,0.11446,-0.02325
5,6,10,1,4.20019,0,10.09286,1,1,10,8.31099,1,0.11752,0.27963,0.09393,-0.02359
6,7,6,0,3.67758,0,10.01138,0,1,6,8.25337,0,0.11837,0.20857,0.13705,0.01868
7,8,10,2,4.94138,3,12.26297,6,4,10,8.30709,2,0.22384,0.57555,0.10707,-0.11677
8,9,10,0,3.52332,0,9.54259,0,3,9,8.31790,2,0.06980,0.12378,0.05778,-0.01202
9,10,10,0,3.96038,0,9.40992,2,3,4,8.31193,1,0.08315,0.10814,0.08383,0.00067


Primary Process-only VM RMSE highest： Lot 8 = 0.22384 µm
Primary Process-only VM RMSE lowest： Lot 1 = 0.06516 µm
Cross-Lot OOD candidates most： Lot 8 (3 wafers)
Lot 8 Process + Sequence RMSE： 0.10707 µm


### 這個輸出代表什麼？

Primary Process-only VM RMSE：

- 最高：**Lot 8 = 0.22384 µm**
- 最低：**Lot 1 = 0.06516 µm**

Cross-Lot OOD candidates 最多的也是：

- **Lot 8：3 片**

Lot 8 另外還有：

- residual review count：**2**
- mean residual Mahalanobis：**4.94138**
- mean OOD Mahalanobis：**12.26297**
- SPC I-chart flagged wafers：**6**
- SPC MR flagged wafers：**4**
- high quality CV count：**2**
- Primary VM max absolute error：**0.57555**

加入 Sequence 後，Lot 8 的 RMSE：

- Process-Only：**0.22384**
- Process + Sequence：**0.10707**
- 變化：**-0.11677**

這也是 10 個 Lots 裡 RMSE 降幅最大的一個。

另外不是每個 Lot 加入 Sequence 都變好：

- Lot 7：RMSE 增加 **0.01868**
- Lot 10：RMSE 增加 **0.00067**

所以這張 Lot-level table 很清楚地顯示，各 Lot 的 prediction error 和 review signals 差異不小。


## 12. 目前距離 Production 還差什麼？

這個專案目前是一個完整的 analysis / modeling prototype，但還不能叫 production-ready。

主要還差：

1. **更多 production data。**  
   現在只有 10 Lots，還不能代表不同 tool、recipe、maintenance state 或長時間 drift。

2. **Chronological / external validation。**  
   Leave-One-Lot-Out 是目前資料內的 unseen-Lot validation，不是未來時間點的真正 deployment test。

3. **Ablation 的外部驗證。**  
   現在已經完成 Process + Sequence nested ablation，但目前仍然只是這 10 Lots 裡的 predictive comparison。

4. **正式 SPC baseline。**  
   現在是 retrospective Phase-I limits，還需要獨立穩定期 production data 建 Phase-II monitoring limits。

5. **OOD calibration validation。**  
   目前是 outer-training-only fixed feature universe + Inner LOGO empirical calibration，還沒有正式 coverage / false-positive-rate guarantee。

6. **DOE / engineering validation。**  
   Process–Quality correlation、bootstrap 和 explainability 都還是 observational / predictive evidence。

7. **Monitoring / retraining policy。**  
   正式上線還需要定義 model drift、sensor drift、missing data、OOD、retraining trigger 和 rollback。


## 13. 輸出 Final Summary Artifacts

最後把 11 真正會給 README、報告或 dashboard 使用的 tables 存成 CSV。

寫出後會再讀回來核對 row count 和欄位，避免最後輸出的檔案和 Notebook 畫面不是同一版。

In [12]:
outputs = {
    PROCESSED_DATA_DIR
    / "11_headline_summary.csv":
        headline,

    PROCESSED_DATA_DIR
    / "11_quality_sequence_summary.csv":
        quality_sequence_summary,

    PROCESSED_DATA_DIR
    / "11_process_quality_candidates.csv":
        process_quality_summary,

    PROCESSED_DATA_DIR
    / "11_vm_ablation_summary.csv":
        ablation_metrics,

    PROCESSED_DATA_DIR
    / "11_priority_cases.csv":
        priority_cases,

    PROCESSED_DATA_DIR
    / "11_final_findings.csv":
        final_findings,

    PROCESSED_DATA_DIR
    / "11_lot_summary.csv":
        lot_final,
}

validation = []

for path, dataframe in outputs.items():
    dataframe.to_csv(
        path,
        index=False,
    )

    check = pd.read_csv(
        path
    )

    if len(check) != len(
        dataframe
    ):
        raise ValueError(
            f"{path.name} row count validation failed"
        )

    if (
        list(
            check.columns
        )
        != list(
            dataframe.columns
        )
    ):
        raise ValueError(
            f"{path.name} schema validation failed"
        )

    validation.append(
        {
            "artifact":
                path.name,
            "rows":
                len(check),
            "schema_match":
                True,
        }
    )

validation_df = pd.DataFrame(
    validation
)

display(
    validation_df
)

print(
    "11 artifact validation：PASS"
)


,artifact,rows,schema_match
0,11_headline_summary.csv,17,True
1,11_quality_sequence_summary.csv,8,True
2,11_process_quality_candidates.csv,12,True
3,11_vm_ablation_summary.csv,4,True
4,11_priority_cases.csv,2,True
5,11_final_findings.csv,10,True
6,11_lot_summary.csv,10,True


11 artifact validation：PASS


### 這個輸出代表什麼？

最後成功寫出 **7 個 Final Summary artifacts**：

1. `11_headline_summary.csv`：**17 rows**
2. `11_quality_sequence_summary.csv`：**8 rows**
3. `11_process_quality_candidates.csv`：**12 rows**
4. `11_vm_ablation_summary.csv`：**4 rows**
5. `11_priority_cases.csv`：**2 rows**
6. `11_final_findings.csv`：**10 rows**
7. `11_lot_summary.csv`：**10 rows**

7 個檔案的：

`schema_match`

全部都是 **True**。

最後也輸出：

`11 artifact validation：PASS`

所以這格確認 11 的最終 CSV 都成功寫出，而且重新讀回後欄位結構沒有改掉。


## 14. 整個專案的最後結果整理

把這份 11 的實際 Output 整理起來，目前可以直接得到下面幾個結果。

### 1. Quality 和 Process 都有 wafer-sequence structure

- Mean Si Etch slope < 0：**10 / 10 Lots**
- CV slope < 0：**9 / 10 Lots**
- Process median `|within-Lot Spearman|`：**0.38788 → 0.08485**
- **24 / 24** process features 在 detrending 後都下降

### 2. Wafer 上有 radial difference

- Mean Si Etch：**43.66812 µm**
- Outer - inner pooled mean：**+5.03895 µm**

### 3. Process–Quality candidates 的 bootstrap uncertainty 不完全一樣

Mean Si Etch：

- Candidates：**5**
- Bootstrap interval 沒跨 0：**4 / 5**

CV Si Etch：

- Candidates：**7**
- Bootstrap interval 沒跨 0：**4 / 7**

### 4. 四種 VM 設定的 RMSE

- Training-Mean：**0.40437**
- Sequence-Only：**0.22093**
- Process-Only：**0.12963**
- Process + Sequence：**0.09540**

Process + Sequence 比 Process-Only：

- RMSE 約低 **26.4%**
- **8 / 10 Lots** 的 RMSE 更低
- Lot 7 和 Lot 10 沒有變好

### 5. Explainability 的兩種結果第一名一致

Grouped permutation：

- 第一名：`PlatenRFTuningCapacitor`
- mean ΔRMSE：**+0.24996**

Final ElasticNet coefficient aggregation：

- 第一名也是 `PlatenRFTuningCapacitor`
- `abs_coefficient_sum = 0.26939`

### 6. SPC、Residual、OOD 和 VM error 的標記不完全重疊

- I-chart feature-wafer flags：**46**
- MR feature-wafer flags：**47**
- Residual candidates：**3**
- Cross-Lot OOD candidates：**5**
- Residual + OOD overlap：**2**

Process score 和 `|OOF error|` 最大的 Spearman：

- `ood_pca_recon_rmse`
- rho = **0.30948**
- 95% interval = **[0.00848, 0.60309]**

### 7. Lot 8 的多個數字都比較突出

Lot 8：

- Primary Process-only RMSE：**0.22384**
- Process + Sequence RMSE：**0.10707**
- Residual candidates：**2**
- OOD candidates：**3**
- SPC I-chart flagged wafers：**6**
- Primary max `|OOF error|`：**0.57555**

其中 Lot 8 的 Process + Sequence RMSE 比 Process-only 低 **0.11677**，是 10 個 Lots 裡降幅最大的一個。

### 8. 最終輸出檔案

11 最後輸出 **7 個 CSV artifacts**，全部：

- `schema_match = True`

而且整體 artifact validation：

- **PASS**

這一版只整理 Notebook 實際跑出的數字與它們之間可以直接比較的差異，沒有另外加入 Output 沒有支持的製程原因或因果結論。
